# Exploratory Transcriptomic Analysis of Psoriasis-Associated Adipose Tissue

## Project Overview

This project is a guided exploratory analysis of a public RNA-seq dataset from cutaneous adipose tissue. The aim is to explore how transcriptomic patterns differ across healthy control, lesional psoriasis, and non-lesional psoriasis samples, while learning how public gene expression data can be grouped, explored, and visualised in Python.


**Research Question:**
How do gene expression patterns differ between healthy control, lesional psoriasis, and non-lesional psoriasis adipose tissue samples?

## 1. Import Required Libraries


In [ ]:
import os #checks files in the working folder
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load and Inspect the Data Files

In [ ]:
#confirm files before starting
print(os.listdir())

In [ ]:
# Load the raw gene count matrix
gene_counts = pd.read_csv("GSE287022_raw_counts_per_gene.csv")

# Take a quick look at the data
gene_counts.head()

In [ ]:
# Number of rows and columns in the dataset
gene_counts.shape

In [ ]:
# Show a sample of the first 10 columns from the dataset
gene_counts.columns[:10]

In [ ]:
# Show the first 5 rows and columns
gene_counts.iloc[:5, :5]

## 3. Extract and Clean Sample Metadata

In [ ]:
# Read the GEO series matrix file
with open("GSE287022_series_matrix.txt", "r") as f:
    lines = f.readlines()

# Keep only the lines that begin with !Sample
sample_lines = [line.strip() for line in lines if line.startswith("!Sample")]

# Show a preview of some of these metadata lines
for line in sample_lines[:25]:
    print(line)

In [ ]:
# Pull the accession ID, titles, source name, and sample characteristics
sample_accession_line = [line for line in sample_lines if line.startswith("!Sample_geo_accession")][0]
sample_title_line = [line for line in sample_lines if line.startswith("!Sample_title")][0]
sample_source_line = [line for line in sample_lines if line.startswith("!Sample_source_name_ch1")][0]
sample_characteristics = [line for line in sample_lines if line.startswith("!Sample_characteristics_ch1")]

In [ ]:
# Create a table showing samples and sample information
sample_ids = sample_accession_line.split("\t")[1:]
sample_titles = sample_title_line.split("\t")[1:]
sample_sources = sample_source_line.split("\t")[1:]

metadata = pd.DataFrame({ "Sample_ID": sample_ids, "Title": sample_titles, "Source": sample_sources })

for i, line in enumerate(sample_characteristics):
    values = line.split("\t")[1:]
    metadata[f"Characteristic_{i+1}"] = values

metadata.head()

In [ ]:
metadata_clean = metadata.copy()

# Clean quotation marks from text columns
text_columns = metadata_clean.select_dtypes(include="object").columns
metadata_clean[text_columns] = metadata_clean[text_columns].apply(lambda col: col.str.strip('"'))

# Extract cleaner labels from the characteristic columns
metadata_clean["Condition"] = metadata_clean["Characteristic_2"].str.replace("condition: ", "", regex=False)
metadata_clean["Timepoint"] = metadata_clean["Characteristic_3"].str.replace("timepoint: ", "", regex=False)
metadata_clean["Tissue"] = metadata_clean["Characteristic_4"].str.replace("tissue: ", "", regex=False)
metadata_clean["Treatment"] = metadata_clean["Characteristic_5"].str.replace("treatment: ", "", regex=False)
metadata_clean["Subject_ID"] = metadata_clean["Characteristic_6"].str.replace("subjectid: ", "", regex=False)

metadata_clean[["Sample_ID", "Title", "Condition", "Timepoint", "Tissue", "Treatment", "Subject_ID"]].head(10)

In [ ]:
# Remove quotation marks from the column names
gene_counts.columns = [col.strip('"') if isinstance(col, str) else col for col in gene_counts.columns]
gene_counts.columns[:10]

In [ ]:
# Skip the first column because it contains gene IDs
sample_columns = list(gene_counts.columns[1:])

# Add the count matrix column names into table
metadata_clean["Count_Column"] = sample_columns

metadata_clean[["Sample_ID", "Title", "Count_Column", "Condition", "Timepoint", "Tissue"]].head(10)

Sample metadata are extracted from the GEO series matrix file and cleaned to identify relevant variables such as condition, tissue type, timepoint, and treatment.

## 4. Define Analysis Groups

In [ ]:
# Isolate specific samples
metadata_filtered = metadata_clean[
    (
        (metadata_clean["Condition"] == "Healthy Vol.") &
        (metadata_clean["Tissue"] == "Normal")
    )
    |
    (
        (metadata_clean["Condition"] == "Psoriasis") &
        (metadata_clean["Timepoint"] == "Baseline") &
        (metadata_clean["Tissue"].isin(["Lesional", "Non-Lesional"]))
    )
].copy()

metadata_filtered[["Sample_ID", "Condition", "Timepoint", "Tissue", "Treatment"]].head(15)

In [ ]:
# Group the samples further
def assign_group(row):
    if row["Condition"] == "Healthy Vol." and row["Tissue"] == "Normal":
        return "Healthy_Control"
    elif row["Condition"] == "Psoriasis" and row["Tissue"] == "Lesional" and row["Timepoint"] == "Baseline":
        return "Psoriasis_Lesional_Baseline"
    elif row["Condition"] == "Psoriasis" and row["Tissue"] == "Non-Lesional" and row["Timepoint"] == "Baseline":
        return "Psoriasis_NonLesional_Baseline"
    else:
        return "Other"

metadata_filtered["Group"] = metadata_filtered.apply(assign_group, axis=1)
metadata_filtered["Group"].value_counts()

## 5. Prepare the Counts Matrix

In [ ]:
# Look for GEO matric IDs in the count matrix columns
match_samples = list(set(metadata_filtered["Sample_ID"]).intersection(set(gene_counts.columns)))
len(match_samples)

In [ ]:
#Show first 10 items from the list match_samples
match_samples[:10]

In [ ]:
# List the number of sample columns and rows in the metadata
num_count_columns = len(gene_counts.columns) - 1
num_metadata_rows = len(metadata_clean)

print("Number of sample columns in counts:", num_count_columns)
print("Number of rows in metadata:", num_metadata_rows)

In [ ]:
# List the sample columns from the count matrix and skip the first column, which contains genes
sample_columns = list(gene_counts.columns[1:])

# Add the count matrix column names into the metadata table
metadata_clean["Count_Column"] = sample_columns

# Preview the cleaned metadata with the linked count columns
metadata_clean[["Sample_ID", "Title", "Count_Column", "Condition", "Timepoint", "Tissue"]].head(10)

In [ ]:
# Keep only the sample groups needed for this analysis
metadata_filtered = metadata_clean[
    (
        (metadata_clean["Condition"] == "Healthy Vol.") &
        (metadata_clean["Tissue"] == "Normal")
    )
    |
    (
        (metadata_clean["Condition"] == "Psoriasis") &
        (metadata_clean["Timepoint"] == "Baseline") &
        (metadata_clean["Tissue"].isin(["Lesional", "Non-Lesional"]))
    )
].copy()

# Preview a few important columns from the filtered metadata
metadata_filtered[["Sample_ID", "Condition", "Timepoint", "Tissue", "Treatment"]].head(15)

In [ ]:
# Assign the correct group label
def assign_group(row):
    if row["Condition"] == "Healthy Vol." and row["Tissue"] == "Normal":
        return "Healthy_Control"
    elif row["Condition"] == "Psoriasis" and row["Tissue"] == "Lesional" and row["Timepoint"] == "Baseline":
        return "Psoriasis_Lesional_Baseline"
    elif row["Condition"] == "Psoriasis" and row["Tissue"] == "Non-Lesional" and row["Timepoint"] == "Baseline":
        return "Psoriasis_NonLesional_Baseline"
    else:
        return "Other"

metadata_filtered["Group"] = metadata_filtered.apply(assign_group, axis=1)
metadata_filtered["Group"].value_counts()

In [ ]:
# Select a column from the table and convert it into a python list
match_samples = metadata_filtered["Count_Column"].tolist()
len(match_samples), match_samples[:10]

In [ ]:
# Take the first column from the count matrix, which contains the gene IDs
gene_column = gene_counts.columns[0]

# Keep the gene ID column plus only the selected sample columns
counts_selected = gene_counts[[gene_column] + match_samples].copy()

# Preview the new filtered count table
counts_selected.head()

## 6. Exploratory Transcriptomic Analysis

This section focuses on exploring the transcriptomic data. To make the count data easier to work with and interpret, the selected expression matrix is prepared, transformed, and then used to identify the genes showing the greatest variability across the sample groups.

In [ ]:
# Show the first few rows of the selected count table again
counts_selected.head()

In [ ]:
# Check the first column and preview its values
gene_column = counts_selected.columns[0]
print("Gene column name:", gene_column)
print(counts_selected[gene_column].head())

In [ ]:
# Converting gene IDs to row labels
expression = counts_selected.set_index(gene_column)
expression.head()

In [ ]:
# Describe the size of the DataFrame
expression.shape

In [ ]:
# Convert all values to numeric
expression = expression.apply(pd.to_numeric, errors="coerce")

# Display data of first few columns
expression.dtypes.head()

In [ ]:
# Check for missing values
expression.isnull().sum().sum()

In [ ]:
# Log transform to reduce skewness, compress large values, and make it easier for visual comparison
expression_log = np.log1p(expression)
expression_log.head()

The raw counts are log-transformed using log1p to make the data less skewed and easier to compare across samples.

In [ ]:
# Count how many samples are present in each final analysis group
metadata_filtered["Group"].value_counts()

In [ ]:
# Set group order
group_order = [
    "Healthy_Control",
    "Psoriasis_NonLesional_Baseline",
    "Psoriasis_Lesional_Baseline"
]

metadata_filtered["Group"] = pd.Categorical(
    metadata_filtered["Group"],
    categories=group_order,
    ordered=True
)

metadata_filtered = metadata_filtered.sort_values("Group")
metadata_filtered[["Sample_ID", "Count_Column", "Group"]].head(15)

In [ ]:
# Create a list of sample columns in the order they appear in the filtered metadata
order_samples = metadata_filtered["Count_Column"].tolist()

# Preview the first 10 ordered sample names
order_samples[:10]

In [ ]:
# Reorder the columns of the log-transformed expression matrix so the samples appear in the same order as the grouped metadata
expression_log = expression_log[order_samples]

# Preview the reordered expression matrix
expression_log.head()

In [ ]:
# Calculating gene variance
gene_variance = expression_log.var(axis=1)

# Sort from highest variance to lowest for the top 30 and return the row labels (gene IDs)
top_variable = gene_variance.sort_values(ascending=False).head(30).index
top_variable

In [ ]:
# Create a smaller matrix containing only the top variable genes
top_expression = expression_log.loc[top_variable]
top_expression.head()

In [ ]:
# Install the mygene package so I can map Ensembl gene IDs to gene symbols
!pip install mygene

In [ ]:
# Import the mygene package
import mygene

# Create an object to query gene information
gene_query = mygene.MyGeneInfo()

In [ ]:
# Turn the top variable gene index into a regular Python list
top_gene_ids = list(top_variable)

# Preview the first 10 gene IDs
top_gene_ids[:10]

In [ ]:
# Query gene information for the top variable genes
gene_info = gene_query.querymany(
    top_gene_ids,
    scopes="ensembl.gene",   # tell mygene that these are Ensembl gene IDs
    fields="symbol,name",    # ask for gene symbol and gene name
    species="human"          # restrict the search to human genes
)

# Preview the first 5 query results
gene_info[:5]

In [ ]:
# Turn the gene query results into a DataFrame and keep only the original query ID, gene symbol, and gene name
mapping_table = pd.DataFrame(gene_info)[["query", "symbol", "name"]]

# View the mapping table
mapping_table

In [ ]:
# Create a dictionary that maps each Ensembl gene ID to its gene symbol
symbol = dict(zip(mapping_table["query"], mapping_table["symbol"]))

# View the dictionary
symbol

In [ ]:
# Make a copy of the top variable gene expression table
top_expression_name = top_expression.copy()

# Replace the row index from Ensembl IDs to gene symbols where available (If a symbol is missing, keep the original gene ID)
top_expression_name.index = [
    symbol.get(gene_id, gene_id) for gene_id in top_expression_name.index
]

# Preview the updated table
top_expression_name.head()

In [ ]:
# Create a larger figure for the heatmap
plt.figure(figsize=(14, 8))

# Plot a heatmap of the top variable genes across the selected samples
sns.heatmap(top_expression_name, cmap="viridis", yticklabels=True, xticklabels=False)

# Add the title and axis labels
plt.title("Top 30 Most Variable Genes Across Selected Samples")
plt.xlabel("Samples")
plt.ylabel("Genes")

# Show the plot
plt.show()

The heatmap of the most variable genes gives an initial view of how gene expression differs across the selected sample groups. The differences seen across healthy control, non-lesional psoriasis, and lesional psoriasis samples may reflect meaningful inflammatory variation.

## 7. Visualisation

This section uses PCA and grouped visualisation to explore whether healthy control, non-lesional psoriasis, and lesional psoriasis samples show distinct gene expression patterns.

In [ ]:
# Reorder the metadata so it matches the sample order in the expression matrix
metadata_plot = metadata_filtered.set_index("Count_Column").loc[order_samples].reset_index()

# Preview the sample columns and their group labels
metadata_plot[["Count_Column", "Group"]].head()

In [ ]:
# Transpose the top expression table so that rows = samples, columns = genes
pca_data = top_expression.T

# Check the shape of the PCA input table
pca_data.shape

In [ ]:
# Import PCA from scikit-learn
from sklearn.decomposition import PCA

In [ ]:
# Create a PCA model that reduces the data into 2 components
pca = PCA(n_components=2)

# Fit the PCA model to the data and get the new PCA coordinates
pca_coordinates = pca.fit_transform(pca_data)

# Turn the PCA coordinates into a DataFrame
pca_table = pd.DataFrame(
    pca_coordinates,
    columns=["PC1", "PC2"],
    index=pca_data.index
).reset_index()

# Rename the sample column
pca_table = pca_table.rename(columns={"index": "Count_Column"})

# Add the group labels from the metadata
pca_table = pca_table.merge(metadata_plot[["Count_Column", "Group"]], on="Count_Column")

# Preview the final PCA table
pca_table.head()

In [ ]:
# See how much of the total variation is explained by PC1 and PC2
pca.explained_variance_ratio_

In [ ]:
# Create a figure for the PCA plot
plt.figure(figsize=(8, 6))

# Plot the samples using PC1 and PC2, coloured by group
sns.scatterplot(data=pca_table, x="PC1", y="PC2", hue="Group", s=70)

# Add the title and axis labels, including the explained variance
plt.title("PCA of Top Variable Genes Across Selected Samples")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")

# Show the plot
plt.show()

The PCA plot gives a simplified view of the variation in gene expression across the selected samples. PC1 explains 82.0% of the variance and PC2 explains 11.8%, which suggests that most of the overall variation is captured in the first two components. While the three groups still show some overlap, the way the samples are distributed suggests meaningful differences across healthy control, non-lesional psoriasis, and lesional psoriasis adipose tissue.

In [ ]:
# Rename the first column of the count matrix to "Ensembl_ID" so the gene ID column is easier to recognise
gene_counts = gene_counts.rename(columns={gene_counts.columns[0]: "Ensembl_ID"})

In [ ]:
# Create shorter group labels for easier plotting
metadata_filtered["Group_Short"] = metadata_filtered["Group"].replace({
    "Healthy_Control": "Healthy",
    "Psoriasis_NonLesional_Baseline": "NonLesional",
    "Psoriasis_Lesional_Baseline": "Lesional"
})

# Preview the original and short group labels
metadata_filtered[["Group", "Group_Short"]].drop_duplicates()

In [ ]:
# Create a mapping from each sample column to its short group label
short_group_map = metadata_filtered.set_index("Count_Column")["Group_Short"]

# Preview the mapping
short_group_map.head()

In [ ]:
# Calculate the average gene expression for each group
group_average_expression = (
    expression_log[order_samples]
    .T
    .join(short_group_map.rename("Group"))
    .groupby("Group")
    .mean()
    .T
)

# Preview the grouped average expression table
group_average_expression.head()

In [ ]:
# Keep only the top variable genes in the grouped expression table
top_group_short = group_average_expression.loc[top_variable].copy()

# Preview the smaller grouped table
top_group_short.head()

In [ ]:
# Replace Ensembl gene IDs with gene symbols where available
top_group_named = top_group_short.copy()
top_group_named.index = [
    symbol.get(gene_id, gene_id) for gene_id in top_group_named.index
]

# Preview the renamed grouped table
top_group_named.head()

In [ ]:
# Plot the average expression of the top variable genes by group
plt.figure(figsize=(8, 10))
sns.heatmap(top_group_named, cmap="viridis", linewidths=0.2)

plt.title("Average Expression of Top Variable Genes by Group")
plt.xlabel("Group")
plt.ylabel("Genes")
plt.tight_layout()
plt.show()

The group-average heatmap gives a clearer summary of the differences in gene expression across Healthy, NonLesional, and Lesional adipose tissue samples. Several highly variable genes showed different average expression patterns across the three groups. While some genes appeared more highly expressed in both psoriasis-associated groups than in healthy controls, others differed between non-lesional and lesional tissue, suggesting meaningful variation within psoriasis-associated adipose tissue.

In [ ]:
# Check which queried genes did not return a gene symbol
mapping_table[mapping_table["symbol"].isna()]

## 8. Key Findings

## Key Findings

- The selected adipose tissue samples show noticeable transcriptomic variation across healthy control, non-lesional psoriasis, and lesional psoriasis groups.
- The PCA shows that most of the overall variation is captured within the first two components, suggesting that the main expression differences can already be seen in this simplified view.
- The grouped heatmap shows that several highly variable genes have different average expression patterns across the three groups.
- Non-lesional and lesional psoriasis-associated adipose tissue do not appear completely identical at the transcriptomic level, which suggests that even non-lesional tissue may still carry distinct molecular changes.
- Overall, this project helped me understand how public RNA-seq data can be organised, filtered, transformed, and visualised to explore gene expression patterns in a disease-related context.